# Exercise 04: nn.Module — Building Neural Network Layers

`nn.Module` is the base class for all neural network components. You define your architecture in `__init__` and the forward pass in `forward()`.

**Instructions:** Fill in the `TODO` markers. Run each cell to verify.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 4.1 Linear Layer

Use `nn.Linear` for a simple linear transformation.

In [3]:
# TODO: Create a linear layer with input features=4 and output features=2
linear = nn.Linear(4, 2)

x = torch.randn(3, 4)  # batch of 3 samples, each with 4 features

# TODO: Pass x through the linear layer
output = linear(x)

assert output.shape == (3, 2)

# nn.Linear has weight and bias parameters
# TODO: Print the shape of the weight matrix
weight_shape = linear.weight.shape

assert weight_shape == (2, 4)
print("4.1 passed!")

4.1 passed!


## 4.2 Simple MLP

Build a 2-layer MLP (multi-layer perceptron) using `nn.Sequential`.

In [4]:
# Input: 10 features, Hidden: 32 neurons, Output: 3 classes
# TODO: Build an MLP with:
#   Linear(10, 32) -> ReLU -> Linear(32, 3)
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 3)
)

x = torch.randn(5, 10)

# TODO: Pass x through the model
output = model(x)

assert output.shape == (5, 3)
print("4.2 passed!")

4.2 passed!


## 4.3 Custom Module

Define a custom `nn.Module` with parameters.

In [5]:
class SimpleAttention(nn.Module):
    """A minimal attention mechanism:
    attention(x) = softmax(x @ W) * x

    W is a learnable (input_dim, input_dim) matrix.
    """

    def __init__(self, input_dim):
        super().__init__()
        # TODO: Define a learnable weight matrix W of shape (input_dim, input_dim)
        # Hint: use nn.Parameter with a random initialization
        self.W = nn.Parameter(torch.randn(input_dim, input_dim))

    def forward(self, x):
        # x shape: (batch, input_dim)
        # TODO: Compute attention weights: softmax(x @ W, dim=-1)
        attn_weights = torch.softmax(x @ self.W, dim=-1)

        # TODO: Return attn_weights * x (element-wise)
        return attn_weights * x

model = SimpleAttention(8)
x = torch.randn(4, 8)

output = model(x)
assert output.shape == (4, 8)

param_count = sum(1 for _ in model.parameters())
assert param_count == 1
print("4.3 passed!")

4.3 passed!


## 4.4 Forward Pass — Tiny CNN

Build a model for image classification and run a forward pass.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 1x28x28 (grayscale, like MNIST)
        # TODO: Define:
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16*7*7, 64)
        self.fc2 = nn.Linear(64, 10)
        pass

    def forward(self, x):
        # x: (batch, 1, 28, 28)
        # TODO: Implement the forward pass:
        #   1. conv1 -> ReLU -> MaxPool2d(2)   -> shape: (B, 8, 14, 14)
        #   2. conv2 -> ReLU -> MaxPool2d(2)   -> shape: (B, 16, 7, 7)
        #   3. Flatten                            -> shape: (B, 16*7*7)
        #   4. fc1 -> ReLU                        -> shape: (B, 64)
        #   5. fc2                                 -> shape: (B, 10)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = TinyCNN()
x = torch.randn(2, 1, 28, 28)
output = model(x)
assert output.shape == (2, 10)
print("4.4 passed!")

4.4 passed!


## 4.5 Loss Functions

Use common loss functions.

In [8]:
# MSE Loss (regression)
pred = torch.tensor([1.5, 2.1, 3.3])
target = torch.tensor([1.0, 2.0, 3.0])

# TODO: Compute MSE loss
mse_loss = nn.MSELoss()(pred, target)

assert abs(mse_loss.item() - 0.1166) < 0.01

# Cross-Entropy Loss (classification)
logits = torch.tensor([[2.0, 0.5, 0.3]])
target_class = torch.tensor([0])

# TODO: Compute cross-entropy loss
ce_loss = nn.CrossEntropyLoss()(logits, target_class)

assert ce_loss.item() > 0

# Binary Cross-Entropy Loss
probs = torch.tensor([0.8, 0.3, 0.9])
targets_binary = torch.tensor([1.0, 0.0, 1.0])

# TODO: Compute binary cross-entropy loss
bce_loss = nn.BCELoss()(probs, targets_binary)

assert bce_loss.item() > 0
print("4.5 passed!")

4.5 passed!


## 4.6 Model Inspection

Inspect model parameters and layers.

In [9]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5),
)

# TODO: Count total number of parameters
total_params = sum(p.numel() for p in model.parameters())

# TODO: Count the number of trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Linear(10, 32) has 10*32 + 32 = 352 parameters
# Linear(32, 5) has 32*5 + 5 = 165 parameters
# Total = 517
assert total_params == 517
assert trainable_params == 517
print("4.6 passed!")

4.6 passed!


## 4.7 Freezing Layers

Freeze layers to prevent gradient updates.

In [10]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 5),
)

# TODO: Freeze the first linear layer (set requires_grad=False for its params)
for param in model[0].parameters():
    param.requires_grad = False

first_layer_frozen = all(not p.requires_grad for p in model[0].parameters())
second_layer_trainable = all(p.requires_grad for p in model[2].parameters())

assert first_layer_frozen == True
assert second_layer_trainable == True

# TODO: Count trainable parameters after freezing
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

assert trainable_params == 165  # only the second layer
print("4.7 passed!")

4.7 passed!
